# Point 4: protocol × model factorial at 25 m/s (RF-Stat, LSTM, BERT)

Same design as `01_protocol_vs_model_15ms.ipynb` (cells A/B/C/D, 20 seeds), 25 m/s regime. Results are cached as JSON in `Detector_Results/factorial`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# Adjust to your Drive layout
P15 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'
P25 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'

def load_and_filter(path, target_speed):
    df = pd.read_csv(path)
    df['label'] = df['Scenario'].map({1: 0, 2: 1, 3: 2})
    df = df.sort_values(['Speed', 'Scenario', 'Time']).reset_index(drop=True)
    df['pseudo_run_id'] = (
        df['Speed'].round(2).astype(str) + '_' +
        df['Scenario'].astype(str) + '_' +
        (df['Time'] // 60).astype(int).astype(str)
    )
    speed_range = {15: (13, 17), 25: (23, 27)}
    lo, hi = speed_range[target_speed]
    return df[(df['Speed'] >= lo) & (df['Speed'] <= hi)].copy()

dfs15 = load_and_filter(P15, 15)
dfs25 = load_and_filter(P25, 25)

print('dfs15:', len(dfs15), 'baris,', dfs15['pseudo_run_id'].nunique(), 'episode')
print('dfs25:', len(dfs25), 'baris,', dfs25['pseudo_run_id'].nunique(), 'episode')

In [ ]:
import shutil, sys

shutil.copy('/content/drive/MyDrive/Detector_Results/pcmef_fixed.py',
            '/content/pcmef_fixed.py')
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import pcmef_fixed as P
print("OK, P siap dipakai")

In [ ]:
dfs15, feasible_15, n_dropped_15 = P.filter_feasible_episodes(dfs15, min_duration_sec=15)
dfs25, feasible_25, n_dropped_25 = P.filter_feasible_episodes(dfs25, min_duration_sec=15)
print(f"15m/s dibuang: {n_dropped_15} episode | 25m/s dibuang: {n_dropped_25} episode")

## RF-Stat, 25 m/s

In [ ]:
OUT = '/content/drive/MyDrive/Detector_Results/factorial'

res25 = P.run_factorial(dfs25, 25, 'RF_Stat', OUT, cells=P.RF_STAT_CELLS)
P.print_factorial(P.decompose_factorial(res25), 'RF-Stat @ 25 m/s')

## LSTM, 25 m/s

In [ ]:
res_lstm25 = P.run_factorial(dfs25, 25, 'LSTM', OUT)
P.print_factorial(P.decompose_factorial(res_lstm25), 'LSTM @ 25 m/s')

## BERT, 25 m/s

In [ ]:
try:
    import torch, transformers
    print('torch:', torch.__version__, '| transformers:', transformers.__version__)
except ImportError:
    print('belum ada, install dulu')

In [ ]:
res_bert25 = P.run_factorial(dfs25, 25, 'BERT', OUT,
                             cells=P.BERT_CELLS, seeds=P.BERT_SEEDS_DEFAULT)
P.print_factorial(P.decompose_factorial(res_bert25), 'BERT @ 25 m/s')
